## EDA has been carried out simultaneously at EDA/merges.ipynb

### Import modules and data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS
from backend.scraping.scrape_reservoirs import get_reservoir_province_rate_limited
from backend.scraping.coordinates_api import get_coordinates_photon
from backend.scraping.real_names_api import get_real_names_nominatim
from backend.data.extract import extract_reservoirs_merged_definitive
from sklearn.metrics.pairwise import haversine_distances

In [ ]:
water_path = PATHS['cleaned_data_notebooks']/ 'water_cleaned.parquet'
water_non_merged_pd = pd.read_parquet(water_path)
water_non_merged_pd.head()

In [ ]:
reservoirs_path = PATHS['cleaned_data_notebooks'] / 'reservoirs_cleaned.csv'
reservoirs_pd = pd.read_csv(reservoirs_path)
reservoirs_pd.head()

In [ ]:
detailed_reservoirs_path = PATHS['cleaned_data_notebooks'] / 'detailed_reservoirs_cleaned.csv'
detailed_reservoirs_pd = pd.read_csv(detailed_reservoirs_path)
detailed_reservoirs_pd.head()

### Create the merged dataframe from reservoirs, with even the non-matching entries

#### Repeated correspondences:

In [ ]:
repeated_in_detailed = ['agrio', 'aguilar campoo', 'algeciras  rambla', 'arcos', 'bachimana alto', 'banos montemayor', 'burguillo', 'castrelo mino', 'chandreja', 'grado i', 'guadalteba', 'ibon ip', 'jerte', 'malpasillo jauja', 'molinos matachel', 'monteagudo vicarias', 'lago negro', 'peares', 'puentes iv', 'sant ponc', 'torre abraham', 'tous', 'vilagudin', 'zahara', 'juan benet']
repeated_in_reservoirs = ['agrio (aznalcollar)', 'aguilar', 'algeciras', 'arcos frontera', 'bachimana (lago)', 'banos', 'burguillo   puente nuevo', 'castrelo', 'chandrexa', 'grado', 'guadalhorce guadalteba', 'ip', 'jerte   plasencia', 'malpasillo ( jauja )', 'molinos', 'vicarias', 'negro (lago)', 'peares  os', 'puentes', 'sant pons', 'torre abrahan', 'tous   ribera', 'villagudin', 'zahara gastor', 'porma (juan benet)']

Now a mapping is created between both names for the same reservoir

In [ ]:
series_repeated = pd.Series(repeated_in_detailed, index=repeated_in_reservoirs)
series_repeated

Converting the names of reservoirs_pd as the names at detailed_reservoirs_pd

In [ ]:
reservoirs_pd['name'] = reservoirs_pd['name'].map(series_repeated).fillna(reservoirs_pd['name'])
reservoirs_pd

We remain with the dataframe reservoirs as the left one, because it has the ID so that it can be paired with water.csv

In [ ]:
final_merged_df = pd.merge(reservoirs_pd, detailed_reservoirs_pd, how='left', on='name')
final_merged_df.head(10)

In [ ]:
print(f"The paired reservoirs of the merged dataframe is {len(final_merged_df[final_merged_df['longitude'].notna()])}")

### Imputing missing values with Scraping

#### Firstly, a function is going to be created so that, if the province of one reservoir has already been scraped, it can be reused.


In [ ]:
saved_df_path = PATHS['pre_EDA'] / 'merges_for_EDA.csv'
saved_df = pd.read_csv(saved_df_path)

In [ ]:
def get_reservoir_province_reusing_data(reservoir_name):
    if saved_df.loc[saved_df['name'] == reservoir_name, 'province'].notna().any():
        return saved_df.loc[saved_df['name'] == reservoir_name, 'province'].values[0]
    return get_reservoir_province_rate_limited(reservoir_name)

In [ ]:
print(f"There are {final_merged_df['province'].isna().sum()} missing values in the 'province' column.")

Imputing missing values using the scraping function

In [ ]:
mask = final_merged_df['province'].isna()
final_merged_df['province'] = final_merged_df['province'].where(final_merged_df['province'].notna(), final_merged_df['name'].apply(get_reservoir_province_reusing_data))

# DO NOT TOUCH

#### Importing the None ones

In [ ]:
final_merged_df[final_merged_df['province'].isnull()]

Looking one by one where they are located:

In [ ]:
mask_None = final_merged_df['province'].isnull()
name_None = ['certescans', 'vellon (pedrezuela)', 'cornalbo', 'lechago']
province_None = ['lleida', 'madrid', 'badajoz', 'teruel']
series_None = pd.Series(province_None, index=name_None)
final_merged_df['province'] = final_merged_df['province'].where(final_merged_df['province'].notna(), final_merged_df['name'].map(series_None))
final_merged_df[mask_None]

# UNTIL HERE

### Unify the province names

First sight:

In [ ]:
final_merged_df['province'].sort_values().unique()

In [ ]:
unified_provinces = {
    'alacant alicante': 'alicante',
    'araba alava': 'alava',
    'castello castellon': 'castellon',
    'gipuzkoa guipuzcoa': 'guipuzcoa',
    'ourense': 'orense',
    'valència valencia': 'valencia',
    'a coruna': 'la coruna',
    'rioja': 'la rioja'
}

final_merged_df['province'] = final_merged_df['province'].map(unified_provinces).fillna(final_merged_df['province'])
final_merged_df['province'].sort_values().unique()

In [ ]:
province_to_community = {
    'alava': 'pais vasco',
    'albacete': 'castilla   mancha',
    'alicante': 'comunitat valenciana',
    'almeria': 'andalucia',
    'asturias': 'principado asturias',
    'avila': 'castilla y leon',
    'badajoz': 'extremadura',
    'barcelona': 'cataluna',
    'burgos': 'castilla y leon',
    'caceres': 'extremadura',
    'cadiz': 'andalucia',
    'cantabria': 'cantabria',
    'castellon': 'comunitat valenciana',
    'ciudad real': 'castilla   mancha',
    'cordoba': 'andalucia',
    'cuenca': 'castilla   mancha',
    'girona': 'cataluna',
    'granada': 'andalucia',
    'guadalajara': 'castilla   mancha',
    'guipuzcoa': 'pais vasco',
    'huelva': 'andalucia',
    'huesca': 'aragon',
    'jaen': 'andalucia',
    'la coruna': 'galicia',
    'la rioja': 'rioja',
    'las palmas': 'canarias',
    'leon': 'castilla y leon',
    'lleida': 'cataluna',
    'lugo': 'galicia',
    'madrid': 'comunidad madrid',
    'malaga': 'andalucia',
    'murcia': 'region murcia',
    'navarra': 'comunidad foral navarra',
    'orense': 'galicia',
    'palencia': 'castilla y leon',
    'pontevedra': 'galicia',
    'salamanca': 'castilla y leon',
    'santa cruz de tenerife': 'canarias',
    'segovia': 'castilla y leon',
    'sevilla': 'andalucia',
    'soria': 'castilla y leon',
    'tarragona': 'cataluna',
    'teruel': 'aragon',
    'toledo': 'castilla   mancha',
    'valencia': 'comunitat valenciana',
    'valladolid': 'castilla y leon',
    'vizcaya': 'pais vasco',
    'zamora': 'castilla y leon',
    'ceuta': 'ceuta',
    'melilla': 'melilla',
    'balears': 'islas baleares',
    'zaragoza': 'aragon'
}

final_merged_df['autonomous_community'] = final_merged_df['province'].map(province_to_community)

### Saving dataframe for EDA

In [ ]:
merges_for_EDA_path = PATHS['pre_EDA'] / 'merges_for_EDA.csv'
merges_for_EDA_path.parent.mkdir(parents=True, exist_ok=True)
final_merged_df.to_csv(merges_for_EDA_path, index=False)

### During EDA workflow

Merge of water with reservoirs:

In [ ]:
water_pd = pd.merge(water_non_merged_pd, final_merged_df[['id', 'capacity', 'crest_elevation', 'province', 'autonomous_community']], on='id', how='left')
water_pd.head()

### Fixing inconsistencies

- With storage higher than capacity:

In [ ]:
mask = water_pd['storage'] > water_pd['capacity']
water_pd.loc[mask, 'storage'] = water_pd.loc[mask, 'capacity']
water_non_merged_pd.loc[mask, 'storage'] = water_pd.loc[mask, 'capacity']

The inconsistencies have also been fixed in the water_non_merged_pd dataframe

- Deleting reservoirs that don't have data up to date: (they turn out to be redundant, there are 374 large reservoirs in Spain as August 2025)

In [ ]:
max_date = water_pd['date'].max()
print(f"The maximum date in the water dataframe is {max_date}.")

In [ ]:
list_final_reservoirs = water_pd[water_pd['date'] == max_date]['id'].values

Updating all the dataframes to ensure consistency:

In [ ]:
water_pd = water_pd[water_pd['id'].isin(list_final_reservoirs)]
water_non_merged_pd = water_non_merged_pd[water_non_merged_pd['id'].isin(list_final_reservoirs)]
reservoirs_pd = reservoirs_pd[reservoirs_pd['id'].isin(list_final_reservoirs)]
final_merged_df = final_merged_df[final_merged_df['id'].isin(list_final_reservoirs)]

### Imputing missing values

In [ ]:
final_merged_df.isna().sum()

### Impute longitude and latitude with a geographical API

In [ ]:
reservoirs_without_coordinates = final_merged_df[final_merged_df['latitude'].isna()]
reservoirs_without_coordinates

#### Define a function so that the API is only used the first time:

In [ ]:
saved_df_path = PATHS['definitive_notebooks'] / 'reservoirs_merged.parquet'
saved_df = pd.read_parquet(saved_df_path)
saved_df

In [ ]:
def get_coordinates_reusing_data(reservoir_list):
    if saved_df['latitude'].isna().any():
        return get_coordinates_photon(reservoir_list)
    else:
        path = PATHS['raw_data_notebooks'] / 'coordinates.csv'
        coordinates = pd.read_csv(path)
        return coordinates

In [ ]:
coordinates = get_coordinates_reusing_data(reservoirs_without_coordinates['name'].values)
coordinates

Saving coordinates so that it's only calculated once

In [ ]:
raw_coordinates_path = PATHS['raw_data_notebooks'] / 'coordinates.csv'
coordinates.to_csv(raw_coordinates_path, index=False)

In [ ]:
coordinates_df = coordinates.copy()
coordinates_df.info()

Seeing which reservoirs weren't completed with the API

In [ ]:
coordinates_df[coordinates_df['latitude'].isna()]

The second reservoir has an issue with it name, we will fix it in the next notebook

In [ ]:
rows = [
        {'name': 'sistema lagos espot', 'latitude': 42.581771, 'longitude': 1.003018},
        {'name': 'agavanzal', 'latitude': 41.979600, 'longitude': -6.200336},
        {'name': 'sistema aguas limpias', 'latitude': 42.793868, 'longitude': -0.329262},
        {'name': 'sistema alto caldares', 'latitude': 42.427684, 'longitude': -0.227609},
        {'name': 'tremp o talarn', 'latitude': 42.204670, 'longitude': 0.949327}
        ]

Filling the missing coordinates by hand

In [ ]:
coordinates_df = pd.concat([pd.DataFrame(rows), coordinates_df], ignore_index=True)
coordinates_df = coordinates_df[coordinates_df['latitude'].notna()]
coordinates_df.set_index('name', inplace=True)
coordinates_df.info()

Fixing the second reservoir name

In [ ]:
final_merged_df.loc[final_merged_df['id'] == 342, 'name'] = 'agavanzal'

Imputing all the coordinates of the reservoirs

In [ ]:
name_to_latitude = coordinates_df['latitude'].to_dict()
name_to_longitude = coordinates_df['longitude'].to_dict()

mask = final_merged_df['name'].isin(coordinates_df.index)
final_merged_df.loc[mask, 'latitude'] = final_merged_df.loc[mask, 'name'].map(name_to_latitude)
final_merged_df.loc[mask, 'longitude'] = final_merged_df.loc[mask, 'name'].map(name_to_longitude)

In [ ]:
final_merged_df.info()

### Impute Crest Elevation, Riverbed and Basin with the closest non null value

In [ ]:
def impute_nearest_neighbour(detailed_reservoirs_data, column_name):
    nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].isna()]
    non_nan_reservoirs = detailed_reservoirs_data[detailed_reservoirs_data[column_name].notna()]

    nan_coord_rads = np.radians(nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)
    non_nan_coord_rads = np.radians(non_nan_reservoirs[['longitude', 'latitude']].to_numpy()).reshape(-1, 2)

    distances = haversine_distances(nan_coord_rads, non_nan_coord_rads) * 6371.0 
    closest_indices = distances.argmin(axis=1)

    # Create a mapping from indices to reservoir values
    reservoir_mapping = non_nan_reservoirs[column_name].iloc[closest_indices]
    detailed_reservoirs_data.loc[nan_reservoirs.index, column_name] = reservoir_mapping.values
    return detailed_reservoirs_data

In [ ]:
final_merged_df = impute_nearest_neighbour(final_merged_df, 'crest_elevation')
final_merged_df = impute_nearest_neighbour(final_merged_df, 'riverbed')
final_merged_df = impute_nearest_neighbour(final_merged_df, 'basin')

Ensure that there are no nulls in these three columns

In [ ]:
final_merged_df.info()

### Mapping reservoirs to their not cleaned names, so that they can be used in production

In [ ]:
def get_real_names_nominatim_reusing_data(final_merged_df):
    path = PATHS['raw_data_notebooks'] / 'real_names.csv'
    try:
        reservoirs = pd.read_csv(path)
        reservoirs = pd.Series(reservoirs['real_name'].values, index=final_merged_df.index)
        final_merged_df['real_name'] = reservoirs
    except:
        final_merged_df = get_real_names_nominatim(final_merged_df)
        final_merged_df['real_name'].to_csv(path, index=False)
    return final_merged_df

In [ ]:
final_merged_df = get_real_names_nominatim_reusing_data(final_merged_df)
final_merged_df

In [ ]:
reservoirs_merged = final_merged_df.copy()

### Drop those duplicated rows by name (there are only two, and they are the same reservoir in both cases)

In [ ]:
reservoirs_merged = reservoirs_merged.drop_duplicates(subset='name', keep='first')

In [ ]:
reservoirs_merged['real_name'].info()

In [ ]:
names_series = reservoirs_merged[['name', 'real_name']]

In [ ]:
accents = {'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ü': 'u', 'ñ': 'n',}
def contains_variations(candidate, real_name):
    real_name_cleaned = real_name.lower()
    for accented_char, unaccented_char in accents.items():
        candidate = candidate.replace(accented_char, unaccented_char)
        real_name_cleaned = real_name_cleaned.replace(accented_char, unaccented_char)

    return (candidate in real_name_cleaned), real_name


In [ ]:
def select_between_commas(names_series):
    definitive = pd.DataFrame(columns=['name', 'real_name'])
    for _, row in names_series.iterrows():
        name = row['name']
        real_name = row['real_name']
        found = False
        name_split = name.split(' ')
        real_name_split = real_name.split(',')
        name_word = 0
        while not found and name_word < len(name_split):
            word = name_split[name_word]
            real_name_word = 0
            while not found and real_name_word < len(real_name_split):
                real_word = real_name_split[real_name_word]
                found, result = contains_variations(word, real_word)
                real_name_word += 1
            name_word += 1
        if found:
            definitive.loc[len(definitive)] = {'name': name, 'real_name': result.strip()}
    return definitive

In [ ]:
definitive = select_between_commas(names_series)

In [ ]:
df = definitive.copy()
stopwords = ['Embalse de la ', 'Embalse del ', 'Embalse de ', 'Embalse ', 'Embassament de la ', 'Embassament de ','Pantano del ', 'Pantano de ', 'Pantano ', 'Presa del ', 'Presa de ', 'Presa das ','Presa ', 'Municipio del ', 'Municipio de ', 'Municipio ', 'Lago del ', 'Lago de ', 'Lago ', 'Camino del ', 'Camino de ', 'Camino ', 'Club Náutico ','Encoro del ', 'Encoro de ', 'Encoro ', 'Carretera del ', 'Carretera de ', 'Bassa de ', 'Estrada dos ', 'Calle ', 'Represa del ', 'toma del ', 'embalse de ', 'Iglesia de ', 'Laguna de ', 'presa de ', 'Pantà de ']
def remove_stopwords(df):
    for index, name in df.iterrows():
        for stopword in stopwords:
            if stopword in name['real_name']:
                df.at[index, 'real_name'] = df['real_name'].str.split(stopword).str[-1].values[index]
                break
    return df

In [ ]:
df = remove_stopwords(df)

In [ ]:
df

In [ ]:
df.loc[164, 'real_name'] = 'Pico de Urdiceto'
df.loc[300, 'real_name'] = 'Gasset'
df.loc[202, 'real_name'] = 'Canal del Taibilla'
df.loc[234, 'real_name'] = 'La Cierva'
df.loc[258, 'real_name'] = 'La Toba'
df.loc[291, 'real_name'] = 'Llosa de Cavall'
df.loc[319, 'real_name'] = 'Las Yeguas'
df.loc[326, 'real_name'] = 'Juan Benet'

In [ ]:
non_matched = [name for name in names_series['name'].values if name not in definitive['name'].values]
non_matched

In [ ]:
missing_real_names = [ {'name': 'chandreja', 'real_name': 'Chandreja'}, {'name': 'ullivarri', 'real_name': 'Ullíbarri-Gamboa'}, {'name': 'montijo', 'real_name': 'Montijo'}, {'name': 'portas', 'real_name': 'Portas'}, {'name': 'llerena', 'real_name': 'Llerena'}, {'name': 'villar rey', 'real_name': 'Villar del Rey'}, {'name': 'alcollarin', 'real_name': 'Alcollarín'}, {'name': 'certescans', 'real_name': 'Lago Certascan'}, {'name': 'zujar', 'real_name': 'Zújar'}, {'name': 'pias (san agustin)', 'real_name': 'Pías (San Agustín)'}, {'name': 'urkulu', 'real_name': 'Urkulu'}, {'name': 'olivargas', 'real_name': 'Olivargas'}, {'name': 'puentes viejas', 'real_name': 'Puentes Viejas'}, {'name': 'parras (las)', 'real_name': 'Las Parras'}, {'name': 'villagonzalo', 'real_name': 'Villagonzalo'}, {'name': 'eiras', 'real_name': 'Eiras'}, {'name': 'ribarroja', 'real_name': 'Ribarroja'}]
df = pd.concat([df, pd.DataFrame(missing_real_names)], ignore_index=True)

In [ ]:
reservoirs_merged['real_name'] = reservoirs_merged['name'].map(df.set_index('name')['real_name'])
reservoirs_merged.head()

### Save all the dataframes modified here

In [ ]:
water_definitive_path = PATHS['definitive_notebooks'] / 'water_definitive.parquet'
water_non_merged_pd.to_parquet(water_definitive_path, index=False)

In [ ]:
merged_reservoirs_path = PATHS['definitive_notebooks'] / 'reservoirs_merged.parquet'
reservoirs_merged.to_parquet(merged_reservoirs_path, index=False)

In [ ]:
reservoirs_definitive_path = PATHS['definitive_notebooks'] / 'reservoirs_definitive.parquet'
reservoirs_pd.to_parquet(reservoirs_definitive_path, index=False)

In [ ]:
detailed_reservoirs_definitive_path = PATHS['definitive_notebooks'] / 'detailed_reservoirs_definitive.parquet'
detailed_reservoirs_pd.to_parquet(detailed_reservoirs_definitive_path, index=False)